In [8]:
pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
pip install boto3 pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("GoogleStock_Dataset_V2.csv")

# Display the first 5 rows
print(df.head())

   Unnamed: 0        Date       Open       High        Low      Close  \
0           0  2013-01-02  17.918339  18.107130  17.846855  18.013729   
1           1  2013-01-03  18.055573  18.229919  17.950716  18.024191   
2           2  2013-01-04  18.165413  18.467529  18.124067  18.380356   
3           3  2013-01-07  18.317591  18.415474  18.196297  18.300158   
4           4  2013-01-08  18.319834  18.338762  18.043119  18.264042   

      Volume  
0  102033017  
1   93075567  
2  110954331  
3   66476239  
4   67295297  


In [12]:
import boto3

s3 = boto3.client('s3', region_name='us-east-1')

bucket_name = "s3-stock-prices-aziz"

try:
    s3.create_bucket(Bucket=bucket_name)
    print(f"Bucket '{bucket_name}' created successfully")
except Exception as e:
    print("Bucket creation failed:")
    print(e)

Bucket 's3-stock-prices-aziz' created successfully


In [13]:
response = s3.list_objects_v2(Bucket=bucket_name)
for obj in response.get('Contents', []):
    print(obj['Key'])

input-data/
output-data/


In [14]:
file_name = "GoogleStock_Dataset_V2.csv"
object_name = "input-data/GoogleStock_Dataset_V2.csv"   # input-data/ prefix zaroori hai

try:
    s3.upload_file(file_name, bucket_name, object_name)
    print(f"Successfully uploaded {file_name} to {bucket_name}")
except Exception as e:
    print("Upload failed:")
    print(e)

Successfully uploaded GoogleStock_Dataset_V2.csv to s3-stock-prices-aziz


In [15]:
input-data/
output-data/
input-data/GoogleStock_Dataset_V2.csv

SyntaxError: invalid syntax (4190706162.py, line 1)

In [18]:
import boto3

sqs = boto3.client('sqs', region_name='us-east-1')

response = sqs.get_queue_url(QueueName='stock-prices-queue')
queue_url = response['QueueUrl']

attrs = sqs.get_queue_attributes(
    QueueUrl=queue_url,
    AttributeNames=['QueueArn']
)
queue_arn = attrs['Attributes']['QueueArn']
print(f"Queue URL: {queue_url}")
print(f"Queue ARN: {queue_arn}")

Queue URL: https://sqs.us-east-1.amazonaws.com/912392607738/stock-prices-queue
Queue ARN: arn:aws:sqs:us-east-1:912392607738:stock-prices-queue


In [19]:
import boto3

s3 = boto3.client('s3', region_name='us-east-1')
bucket_name = "s3-stock-prices-aziz"

response = s3.list_objects_v2(Bucket=bucket_name)

In [20]:
for obj in response.get('Contents', []):
    print(obj['Key'])

input-data/
input-data/GoogleStock_Dataset_V2.csv
output-data/


In [21]:
sqs = boto3.client('sqs', region_name='us-east-1')

q_response = sqs.get_queue_url(QueueName="stock-prices-queue")
queue_url = q_response['QueueUrl']
print(queue_url)

https://sqs.us-east-1.amazonaws.com/912392607738/stock-prices-queue


In [22]:
attrs = sqs.get_queue_attributes(
    QueueUrl=queue_url,
    AttributeNames=['QueueArn', 'VisibilityTimeout', 'ReceiveMessageWaitTimeSeconds']
)
print(attrs['Attributes'])

{'QueueArn': 'arn:aws:sqs:us-east-1:912392607738:stock-prices-queue', 'VisibilityTimeout': '90', 'ReceiveMessageWaitTimeSeconds': '10'}


In [23]:
iam = boto3.client('iam')

role = iam.get_role(RoleName="role-stock-prices")
print(role['Role']['Arn'])
print(role['Role']['AssumeRolePolicyDocument'])

arn:aws:iam::912392607738:role/role-stock-prices
{'Version': '2012-10-17', 'Statement': [{'Effect': 'Allow', 'Principal': {'Service': 'lambda.amazonaws.com'}, 'Action': 'sts:AssumeRole'}]}


In [24]:
inline_policies = iam.list_role_policies(RoleName="role-stock-prices")
print(inline_policies['PolicyNames'])

['role-stock-pricesPolicy']


In [25]:
policy_detail = iam.get_role_policy(
    RoleName="role-stock-prices",
    PolicyName="role-stock-pricesPolicy"
)
print(policy_detail['PolicyDocument'])

{'Version': '2012-10-17', 'Statement': [{'Effect': 'Allow', 'Action': 's3:GetObject', 'Resource': 'arn:aws:s3:::s3-stock-prices-aziz/input-data/*'}, {'Effect': 'Allow', 'Action': 's3:PutObject', 'Resource': 'arn:aws:s3:::s3-stock-prices-aziz/output-data/*'}, {'Effect': 'Allow', 'Action': ['sqs:ReceiveMessage', 'sqs:DeleteMessage', 'sqs:GetQueueAttributes'], 'Resource': 'arn:aws:sqs:us-east-1:912392607738:stock-prices-queue'}, {'Effect': 'Allow', 'Action': ['logs:CreateLogGroup', 'logs:CreateLogStream', 'logs:PutLogEvents'], 'Resource': 'arn:aws:logs:*:*:*'}]}


In [26]:
event = {
    "Records": [
        {
            "messageId": "e1f2a3b4-...",
            "receiptHandle": "AQEB...",
            "body": "{\"Records\":[{\"eventVersion\":\"2.1\",\"eventSource\":\"aws:s3\",\"awsRegion\":\"us-east-1\",\"eventTime\":\"2026-07-14T10:15:00.000Z\",\"eventName\":\"ObjectCreated:Put\",\"s3\":{\"bucket\":{\"name\":\"s3-stock-prices-aziz\",\"arn\":\"arn:aws:s3:::s3-stock-prices-aziz\"},\"object\":{\"key\":\"input-data/GoogleStock_Dataset_V2.csv\",\"size\":102400}}}]}",
            "attributes": {...},
            "eventSource": "aws:sqs",
            "eventSourceARN": "arn:aws:sqs:us-east-1:912392607738:stock-prices-queue"
        }
    ]
}

In [28]:
import json

# Sample event jaisa AWS Lambda ko SQS se milta hai (testing ke liye manually bana rahe hain)
event = {
    "Records": [
        {
            "messageId": "test-123",
            "body": json.dumps({
                "Records": [
                    {
                        "s3": {
                            "bucket": {"name": "s3-stock-prices-aziz"},
                            "object": {"key": "input-data/GoogleStock_Dataset_V2.csv"}
                        }
                    }
                ]
            })
        }
    ]
}

In [29]:
for record in event['Records']:
    s3_event = json.loads(record['body'])
    bucket_name = s3_event['Records'][0]['s3']['bucket']['name']
    object_key = s3_event['Records'][0]['s3']['object']['key']
    print("Bucket:", bucket_name)
    print("Key:", object_key)

Bucket: s3-stock-prices-aziz
Key: input-data/GoogleStock_Dataset_V2.csv


In [30]:
import boto3
import pandas as pd

def download_and_read_csv(bucket, key):
    s3_client = boto3.client('s3')
    response = s3_client.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(response['Body'])
    return df

In [31]:
df = download_and_read_csv(bucket_name, object_key)
print(df.shape)
print(df.head())

(2775, 7)
   Unnamed: 0        Date       Open       High        Low      Close  \
0           0  2013-01-02  17.918339  18.107130  17.846855  18.013729   
1           1  2013-01-03  18.055573  18.229919  17.950716  18.024191   
2           2  2013-01-04  18.165413  18.467529  18.124067  18.380356   
3           3  2013-01-07  18.317591  18.415474  18.196297  18.300158   
4           4  2013-01-08  18.319834  18.338762  18.043119  18.264042   

      Volume  
0  102033017  
1   93075567  
2  110954331  
3   66476239  
4   67295297  


In [32]:
from datetime import datetime

def transform_data(df):
    # 1. Unnamed index column hatao
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    
    # 2. symbol column add karo
    df['symbol'] = 'GOOG'
    
    # 3. close_pct_change calculate karo
    df['close_pct_change'] = ((df['Close'] - df['Open']) / df['Open']) * 100
    
    # 4. Volume clean karo (low volume ko N/A karo)
    df['Volume'] = df['Volume'].apply(lambda v: 'N/A' if v < 100000 else v)
    
    # 5. created_at timestamp add karo
    df['created_at'] = datetime.now().isoformat()
    
    return df

In [33]:
transformed_df = transform_data(df)
print(transformed_df.shape)
print(transformed_df.head())

(2775, 9)
         Date       Open       High        Low      Close     Volume symbol  \
0  2013-01-02  17.918339  18.107130  17.846855  18.013729  102033017   GOOG   
1  2013-01-03  18.055573  18.229919  17.950716  18.024191   93075567   GOOG   
2  2013-01-04  18.165413  18.467529  18.124067  18.380356  110954331   GOOG   
3  2013-01-07  18.317591  18.415474  18.196297  18.300158   66476239   GOOG   
4  2013-01-08  18.319834  18.338762  18.043119  18.264042   67295297   GOOG   

   close_pct_change                  created_at  
0          0.532361  2026-07-14T11:23:39.063842  
1         -0.173806  2026-07-14T11:23:39.063842  
2          1.183254  2026-07-14T11:23:39.063842  
3         -0.095172  2026-07-14T11:23:39.063842  
4         -0.304543  2026-07-14T11:23:39.063842  


In [34]:
from io import StringIO

def upload_csv_to_s3(df, bucket, input_key):
    s3_client = boto3.client('s3')
    
    # input-data/ ko output-data/ se replace karo
    output_key = input_key.replace('input-data/', 'output-data/')
    
    # DataFrame ko CSV string mein convert karo (memory mein)
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)
    
    # S3 pe upload karo
    s3_client.put_object(
        Bucket=bucket,
        Key=output_key,
        Body=csv_buffer.getvalue()
    )
    
    return output_key

In [35]:
output_key = upload_csv_to_s3(transformed_df, bucket_name, object_key)
print(f"Uploaded to: {output_key}")

Uploaded to: output-data/GoogleStock_Dataset_V2.csv


In [36]:
response = s3.list_objects_v2(Bucket=bucket_name, Prefix="output-data/")
for obj in response.get('Contents', []):
    print(obj['Key'], "-", obj['Size'], "bytes")

output-data/ - 0 bytes
output-data/GoogleStock_Dataset_V2.csv - 399959 bytes


In [37]:
import json
import boto3
import pandas as pd
from io import StringIO
from datetime import datetime

s3_client = boto3.client('s3')


def download_and_read_csv(bucket, key):
    response = s3_client.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(response['Body'])
    return df


def transform_data(df):
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    df['symbol'] = 'GOOG'
    df['close_pct_change'] = ((df['Close'] - df['Open']) / df['Open']) * 100
    df['Volume'] = df['Volume'].apply(lambda v: 'N/A' if v < 100000 else v)
    df['created_at'] = datetime.now().isoformat()

    return df


def upload_csv_to_s3(df, bucket, input_key):
    output_key = input_key.replace('input-data/', 'output-data/')

    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    s3_client.put_object(
        Bucket=bucket,
        Key=output_key,
        Body=csv_buffer.getvalue()
    )

    return output_key


def lambda_handler(event, context):
    processed_files = []
    failed_files = []

    for record in event['Records']:
        try:
            # Step 1: SQS body parse karo -> S3 event nikalo
            s3_event = json.loads(record['body'])
            bucket_name = s3_event['Records'][0]['s3']['bucket']['name']
            object_key = s3_event['Records'][0]['s3']['object']['key']

            print(f"Processing file: {object_key} from bucket: {bucket_name}")

            # Step 2: CSV download + read
            df = download_and_read_csv(bucket_name, object_key)
            print(f"Downloaded CSV with shape: {df.shape}")

            # Step 3: Transform
            transformed_df = transform_data(df)
            print(f"Transformation complete: {transformed_df.shape}")

            # Step 4: Upload processed file
            output_key = upload_csv_to_s3(transformed_df, bucket_name, object_key)
            print(f"Uploaded processed file to: {output_key}")

            processed_files.append(output_key)

        except Exception as e:
            print(f"ERROR processing record: {str(e)}")
            failed_files.append(str(e))

    return {
        "statusCode": 200 if not failed_files else 500,
        "body": json.dumps({
            "processed": processed_files,
            "failed": failed_files
        })
    }

In [1]:
import boto3

s3 = boto3.client('s3', region_name='us-east-1')
bucket_name = "s3-stock-prices-aziz"

s3.upload_file("GoogleStock_Dataset_V2.csv", bucket_name, "input-data/test_run_1.csv")
print("Uploaded — ab 15-20 seconds wait karte hain pipeline chalne ka")

Uploaded — ab 15-20 seconds wait karte hain pipeline chalne ka


In [2]:
import time
time.sleep(20)

response = s3.list_objects_v2(Bucket=bucket_name, Prefix="output-data/")
for obj in response.get('Contents', []):
    print(obj['Key'], "-", obj['LastModified'])

output-data/ - 2026-07-14 05:00:16+00:00
output-data/GoogleStock_Dataset_V2.csv - 2026-07-14 06:24:35+00:00


In [3]:
response = s3.list_objects_v2(Bucket=bucket_name, Prefix="input-data/")
for obj in response.get('Contents', []):
    print(obj['Key'], "-", obj['LastModified'])


input-data/ - 2026-07-14 04:59:52+00:00
input-data/GoogleStock_Dataset_V2.csv - 2026-07-14 05:01:22+00:00
input-data/test_run_1.csv - 2026-07-14 10:53:01+00:00


In [4]:
s3.upload_file("GoogleStock_Dataset_V2.csv", bucket_name, "input-data/test_run_2.csv")
print("Uploaded test_run_2.csv")

Uploaded test_run_2.csv


In [5]:
import time
time.sleep(25)

response = s3.list_objects_v2(Bucket=bucket_name, Prefix="output-data/")
for obj in response.get('Contents', []):
    print(obj['Key'], "-", obj['LastModified'])

output-data/ - 2026-07-14 05:00:16+00:00
output-data/GoogleStock_Dataset_V2.csv - 2026-07-14 06:24:35+00:00


In [6]:
s3.upload_file("GoogleStock_Dataset_V2.csv", bucket_name, "input-data/test_run_3.csv")
print("Uploaded test_run_3.csv")

Uploaded test_run_3.csv


In [7]:
import time
time.sleep(30)

response = s3.list_objects_v2(Bucket=bucket_name, Prefix="output-data/")
for obj in response.get('Contents', []):
    print(obj['Key'], "-", obj['LastModified'])

output-data/ - 2026-07-14 05:00:16+00:00
output-data/GoogleStock_Dataset_V2.csv - 2026-07-14 06:24:35+00:00
output-data/test_run_1.csv - 2026-07-14 11:14:04+00:00
output-data/test_run_2.csv - 2026-07-14 11:13:31+00:00
output-data/test_run_3.csv - 2026-07-14 11:14:45+00:00


In [8]:
df_check = download_and_read_csv(bucket_name, "output-data/test_run_3.csv")
print(df_check.shape)
print(df_check.head())

NameError: name 'download_and_read_csv' is not defined

In [9]:
import boto3
import pandas as pd

s3 = boto3.client('s3', region_name='us-east-1')

def download_and_read_csv(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(response['Body'])
    return df

bucket_name = "s3-stock-prices-aziz"

df_check = download_and_read_csv(bucket_name, "output-data/test_run_3.csv")
print(df_check.shape)
print(df_check.head())

(2775, 9)
         Date       Open       High        Low      Close     Volume symbol  \
0  2013-01-02  17.918339  18.107130  17.846855  18.013729  102033017   GOOG   
1  2013-01-03  18.055573  18.229919  17.950716  18.024191   93075567   GOOG   
2  2013-01-04  18.165413  18.467529  18.124067  18.380356  110954331   GOOG   
3  2013-01-07  18.317591  18.415474  18.196297  18.300158   66476239   GOOG   
4  2013-01-08  18.319834  18.338762  18.043119  18.264042   67295297   GOOG   

   close_pct_change                  created_at  
0          0.532361  2026-07-14T11:14:44.054203  
1         -0.173806  2026-07-14T11:14:44.054203  
2          1.183254  2026-07-14T11:14:44.054203  
3         -0.095172  2026-07-14T11:14:44.054203  
4         -0.304543  2026-07-14T11:14:44.054203  
